# Brick 1 — Pure ecological model (deep dive)

Interactive exploration of the Lotka-Volterra dynamics of the `bilevel-fishery` framework.

**Objectives**:
1. Free trajectory and phase portrait (closed orbits around the L-V centre).
2. Multiple initial conditions: concentric orbits.
3. Flow field $(dF/dt, dA/dt)$: the vector field that generates the orbits.
4. Moderate fishing: the system finds a new equilibrium.
5. Excessive fishing: collapse and `EcologyInstabilityError`.
6. Harvest sweep: equilibrium and time-to-collapse as functions of $H$.
7. Conserved quantity (L-V invariant): how does Euler drift, how well does RK45 preserve it?
8. Parameter sensitivity: what does each Greek parameter ($\alpha, \beta, \delta, \gamma$) actually do?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from bilevel_fishery.ecology import (
    EcologicalState,
    EcologyInstabilityError,
    EcologyParams,
    step,
)

plt.rcParams["figure.figsize"] = (10, 5)

## 1. Free trajectory (no fishing)

At default parameters, the equilibrium is at $(F^*, A^*) = (\alpha/\beta,\ \gamma/\delta) = (10,\ 20)$. We start slightly off-equilibrium to see the oscillations.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")

state = EcologicalState(fish=15.0, algae=15.0)
fish_traj = [state.fish]
algae_traj = [state.algae]
for _ in range(400):
    state = step(state, params, harvest=0.0)
    fish_traj.append(state.fish)
    algae_traj.append(state.algae)

t = np.arange(len(fish_traj)) * params.dt
f_star = params.alpha / params.beta
a_star = params.gamma / params.delta

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(t, fish_traj, label="fish (predator)")
ax1.plot(t, algae_traj, label="algae (prey)")
ax1.axhline(f_star, color="C0", ls=":", label=r"$F^* = \alpha/\beta$")
ax1.axhline(a_star, color="C1", ls=":", label=r"$A^* = \gamma/\delta$")
ax1.set_xlabel("time")
ax1.set_ylabel("biomass")
ax1.set_title("Time series (no harvest)")
ax1.legend()

ax2.plot(algae_traj, fish_traj, lw=0.8)
ax2.plot([a_star], [f_star], "r*", ms=18, label="equilibrium")
ax2.set_xlabel("algae")
ax2.set_ylabel("fish")
ax2.set_title("Phase portrait (closed orbit)")
ax2.legend()
plt.tight_layout()
plt.show()

## 2. Phase portrait grid — concentric orbits around the centre

Without harvest, every initial condition gives a **closed orbit** in $(A, F)$ space. The orbits are concentric around the centre $(A^*, F^*)$, with size set by the initial perturbation. The L-V centre is **neutrally stable**: orbits neither shrink nor grow.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")
f_star = params.alpha / params.beta
a_star = params.gamma / params.delta

scales = [0.4, 0.55, 0.7, 0.85, 1.15, 1.3, 1.5]
off_radial = [(8.0, 30.0), (14.0, 25.0), (5.0, 12.0), (17.0, 13.0)]

fig, ax = plt.subplots(figsize=(8, 8))
cmap = plt.get_cmap("plasma")

for s, color in zip(scales, cmap(np.linspace(0.1, 0.9, len(scales))), strict=True):
    state = EcologicalState(fish=f_star * s, algae=a_star * s)
    fish_traj, algae_traj = [state.fish], [state.algae]
    for _ in range(800):
        state = step(state, params, harvest=0.0)
        fish_traj.append(state.fish)
        algae_traj.append(state.algae)
    ax.plot(
        algae_traj,
        fish_traj,
        color=color,
        alpha=0.85,
        lw=1.4,
        label=f"IC = {s:.2f}·(F*, A*)",
    )
    ax.plot(algae_traj[0], fish_traj[0], "o", color=color, ms=5)

for f0, a0 in off_radial:
    state = EcologicalState(fish=f0, algae=a0)
    fish_traj, algae_traj = [state.fish], [state.algae]
    for _ in range(800):
        state = step(state, params, harvest=0.0)
        fish_traj.append(state.fish)
        algae_traj.append(state.algae)
    ax.plot(algae_traj, fish_traj, color="gray", alpha=0.4, lw=1, ls="--")

ax.plot(
    a_star, f_star, "r*", ms=22, label=f"centre ({a_star:.0f}, {f_star:.0f})", zorder=10
)
ax.set_xlabel("algae A")
ax.set_ylabel("fish F")
ax.set_title("Phase portrait — closed orbits around the L-V centre")
ax.grid(alpha=0.3)
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Flow field — the vector field $(dF/dt, dA/dt)$

Each arrow shows the local flow direction at $(A, F)$. The colour encodes the magnitude $\|(dF/dt, dA/dt)\|$. The flow circulates **counter-clockwise** around the centre — the same direction the phase portrait orbits travel.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")
f_star = params.alpha / params.beta
a_star = params.gamma / params.delta

F_grid, A_grid = np.meshgrid(np.linspace(1, 22, 22), np.linspace(2, 40, 24))
dF = params.delta * A_grid * F_grid - params.gamma * F_grid
dA = params.alpha * A_grid - params.beta * A_grid * F_grid
magnitude = np.sqrt(dF**2 + dA**2)

fig, ax = plt.subplots(figsize=(10, 7))
quiv = ax.quiver(
    A_grid,
    F_grid,
    dA / (magnitude + 1e-8),
    dF / (magnitude + 1e-8),
    magnitude,
    cmap="viridis",
    scale=35,
    alpha=0.85,
)
plt.colorbar(quiv, ax=ax, label=r"$\|(dF/dt,\ dA/dt)\|$")

state = EcologicalState(fish=15.0, algae=15.0)
fish_traj, algae_traj = [state.fish], [state.algae]
for _ in range(800):
    state = step(state, params, harvest=0.0)
    fish_traj.append(state.fish)
    algae_traj.append(state.algae)
ax.plot(
    algae_traj,
    fish_traj,
    color="red",
    lw=2,
    alpha=0.9,
    label="example orbit ($F_0 = A_0 = 15$)",
)
ax.plot(a_star, f_star, "r*", ms=22, label="centre", zorder=10)

ax.set_xlabel("algae A")
ax.set_ylabel("fish F")
ax.set_title(r"Flow field $(dA/dt,\ dF/dt)$ — arrows = direction, colour = magnitude")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

## 4. Moderate fishing: the system finds a new equilibrium

With fishing pressure below the endogenous production, the fish stock decreases then oscillates around a lower equilibrium. Algae grow slightly to compensate.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")
harvests = [0.0, 0.2, 0.4]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, harvest in zip(axes, harvests, strict=True):
    state = EcologicalState(fish=15.0, algae=15.0)
    fish_traj = [state.fish]
    algae_traj = [state.algae]
    for _ in range(400):
        state = step(state, params, harvest=harvest)
        fish_traj.append(state.fish)
        algae_traj.append(state.algae)
    t = np.arange(len(fish_traj)) * params.dt
    ax.plot(t, fish_traj, label="fish")
    ax.plot(t, algae_traj, label="algae")
    ax.set_xlabel("time")
    ax.set_title(f"harvest = {harvest}")
    ax.legend()
axes[0].set_ylabel("biomass")
plt.tight_layout()
plt.show()

## 5. Excessive fishing: collapse and `EcologyInstabilityError`

When fishing pressure exceeds endogenous production, the stock collapses towards 0. The RK45 integrator eventually produces a `fish < 0` (non-physical territory) and `step()` **refuses to continue** by raising an `EcologyInstabilityError` — a deliberate fail-loud, in contrast with the silent clamp of the master codebase.

In [ ]:
params = EcologyParams(dt=0.05, integrator="rk45")
state = EcologicalState(fish=15.0, algae=15.0)
fish_traj, algae_traj = [state.fish], [state.algae]

crashed_at: int | None = None
max_steps = 500
for i in range(max_steps):
    try:
        state = step(state, params, harvest=2.0)
    except EcologyInstabilityError as err:
        crashed_at = i
        print(f"Crashed at step {i} (t={i * params.dt:.2f}): {err}")
        break
    fish_traj.append(state.fish)
    algae_traj.append(state.algae)

t = np.arange(len(fish_traj)) * params.dt
fig, ax = plt.subplots()
ax.plot(t, fish_traj, label="fish")
ax.plot(t, algae_traj, label="algae")
ax.set_xlabel("time")
ax.set_ylabel("biomass")
if crashed_at is not None:
    ax.axvline(
        crashed_at * params.dt,
        color="red",
        ls="--",
        label=f"crash @ t={crashed_at * params.dt:.2f}",
    )
ax.set_title("Over-harvest (H=2.0): collapse")
ax.legend()
plt.show()

## 6. Harvest sweep — equilibrium and time-to-collapse vs $H$

How does the system respond to different constant harvest rates? Two views:

- **Long-term mean biomass** (averaged over the last quarter of the run when the trajectory survives): how much fish and algae remain on average.
- **Time-to-collapse**: when does the fish biomass first drop below a near-zero threshold (here 1.0)? `EcologyInstabilityError` is treated as instant collapse.

In [ ]:
H_values = np.linspace(0.0, 2.5, 26)
N_LONG = 1500
DT = 0.05
COLLAPSE_THRESHOLD = 1.0

F_mean = np.zeros(len(H_values))
A_mean = np.zeros(len(H_values))
t_to_collapse = np.zeros(len(H_values))

for i, H in enumerate(H_values):
    params = EcologyParams(dt=DT, integrator="rk45")
    state = EcologicalState(fish=15.0, algae=15.0)
    fish_arr, algae_arr = [], []
    collapse_step = None
    for j in range(N_LONG):
        try:
            state = step(state, params, harvest=H)
        except EcologyInstabilityError:
            collapse_step = j
            break
        fish_arr.append(state.fish)
        algae_arr.append(state.algae)
        if collapse_step is None and state.fish < COLLAPSE_THRESHOLD:
            collapse_step = j
    fish_arr = np.array(fish_arr) if fish_arr else np.array([0.0])
    algae_arr = np.array(algae_arr) if algae_arr else np.array([0.0])
    if len(fish_arr) >= 400:
        F_mean[i] = fish_arr[-300:].mean()
        A_mean[i] = algae_arr[-300:].mean()
    else:
        F_mean[i] = fish_arr.mean()
        A_mean[i] = algae_arr.mean()
    t_to_collapse[i] = (collapse_step if collapse_step is not None else N_LONG) * DT

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(H_values, F_mean, "o-", label="fish (long-term mean)", lw=2)
ax1.plot(H_values, A_mean, "o-", label="algae (long-term mean)", lw=2)
ax1.set_xlabel("constant harvest H")
ax1.set_ylabel("biomass")
ax1.set_title("Long-term mean biomass vs harvest pressure")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(H_values, t_to_collapse, "o-", color="C3", lw=2)
ax2.axhline(
    N_LONG * DT, color="green", ls="--", alpha=0.6, label="never collapsed in horizon"
)
ax2.set_xlabel("constant harvest H")
ax2.set_ylabel(f"time until fish < {COLLAPSE_THRESHOLD}")
ax2.set_title("Time to collapse")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Euler vs RK45: amplitude divergence

For a moderate `dt`, explicit Euler accumulates amplitude error: oscillations grow artificially. RK45 stays faithful to the closed orbit because the L-V system without harvest is energy-conserving.

In [ ]:
dts = [0.05, 0.2]


def integrate(params: EcologyParams, n_steps: int) -> list[float]:
    """Return fish trajectory; stop early if EcologyInstabilityError fires."""
    state = EcologicalState(fish=15.0, algae=15.0)
    traj = [state.fish]
    for _ in range(n_steps):
        try:
            state = step(state, params, harvest=0.0)
        except EcologyInstabilityError:
            break
        traj.append(state.fish)
    return traj


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, dt in zip(axes, dts, strict=True):
    p_euler = EcologyParams(dt=dt, integrator="euler")
    p_rk45 = EcologyParams(dt=dt, integrator="rk45")
    n_steps = int(20 / dt)
    fish_e = integrate(p_euler, n_steps)
    fish_r = integrate(p_rk45, n_steps)
    ax.plot(np.arange(len(fish_e)) * dt, fish_e, label="Euler", lw=1.5)
    ax.plot(np.arange(len(fish_r)) * dt, fish_r, label="RK45", lw=1.5)
    ax.set_title(f"fish biomass, dt = {dt}")
    ax.set_xlabel("time")
    ax.legend()
axes[0].set_ylabel("fish")
plt.tight_layout()
plt.show()

## 8. Conserved quantity — L-V invariant under Euler vs RK45

The harvest-free Lotka-Volterra system has a conserved quantity (a first integral):

$$V(F, A) = \delta A - \gamma \log A + \beta F - \alpha \log F$$

(With $F$ = predator and $A$ = prey, as in our convention.) Along any orbit, $V$ is *constant* — the orbits in phase space are level curves of $V$. A faithful integrator must keep $V$ constant; numerical errors show up as drift in $V$ over time.

In [ ]:
def lv_invariant(F: float, A: float, p: EcologyParams) -> float:
    """V(F, A) = δA - γ·logA + βF - α·logF, conserved by the exact L-V flow.

    Derivation: with F = predator, A = prey, V is the first integral of
    dF/dt = δAF - γF, dA/dt = αA - βAF (dV/dt ≡ 0 along orbits).
    """
    return p.delta * A - p.gamma * np.log(A) + p.beta * F - p.alpha * np.log(F)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
configs = [
    ("dt=0.05, RK45", EcologyParams(dt=0.05, integrator="rk45"), "C0", "-"),
    ("dt=0.05, Euler", EcologyParams(dt=0.05, integrator="euler"), "C1", "-"),
    ("dt=0.2, RK45", EcologyParams(dt=0.2, integrator="rk45"), "C0", "--"),
    ("dt=0.2, Euler", EcologyParams(dt=0.2, integrator="euler"), "C1", "--"),
]

for label, p, color, ls in configs:
    state = EcologicalState(fish=15.0, algae=15.0)
    V_traj = [lv_invariant(state.fish, state.algae, p)]
    times = [0.0]
    n_steps = int(40 / p.dt)
    for k in range(n_steps):
        try:
            state = step(state, p, harvest=0.0)
        except EcologyInstabilityError:
            break
        V_traj.append(lv_invariant(state.fish, state.algae, p))
        times.append((k + 1) * p.dt)
    V_arr = np.array(V_traj)
    t_arr = np.array(times)
    axes[0].plot(t_arr, V_arr, color=color, ls=ls, lw=1.5, label=label)
    rel = (V_arr - V_arr[0]) / abs(V_arr[0]) * 100
    axes[1].plot(t_arr, rel, color=color, ls=ls, lw=1.5, label=label)

axes[0].set_xlabel("time")
axes[0].set_ylabel(r"$V(F, A)$")
axes[0].set_title("L-V invariant — exact dynamics keep V constant")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_xlabel("time")
axes[1].set_ylabel(r"relative drift $(V - V_0)\,/\,|V_0|$  [%]")
axes[1].set_title("Drift of V over time (smaller is better)")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Parameter sensitivity — what does each Greek do?

The L-V dynamics are governed by four positive parameters. To get intuition, we sweep each one independently around its default and plot the resulting fish trajectory (no harvest).

- $\alpha$: intrinsic algae growth rate. Larger → faster prey replenishment.
- $\beta$: predation rate. Larger → predator pressure on algae is stronger.
- $\delta$: fish growth efficiency per algae consumed. Larger → fish recover faster.
- $\gamma$: natural fish mortality. Larger → fish die faster.

The equilibrium itself shifts: $F^* = \alpha/\beta$, $A^* = \gamma/\delta$.

In [ ]:
defaults = EcologyParams()
sweeps = {
    "alpha": (defaults.alpha, [0.5, 1.0, 1.5, 2.0]),
    "beta": (defaults.beta, [0.05, 0.10, 0.15, 0.20]),
    "delta": (defaults.delta, [0.04, 0.075, 0.12, 0.18]),
    "gamma": (defaults.gamma, [0.8, 1.5, 2.2, 3.0]),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (pname, (base, values)) in zip(axes.flat, sweeps.items(), strict=True):
    cmap = plt.get_cmap("plasma")
    for j, v in enumerate(values):
        p = EcologyParams(dt=0.05, integrator="rk45", **{pname: v})
        state = EcologicalState(fish=15.0, algae=15.0)
        traj = [state.fish]
        for _ in range(400):
            try:
                state = step(state, p, harvest=0.0)
            except EcologyInstabilityError:
                break
            traj.append(state.fish)
        t = np.arange(len(traj)) * p.dt
        is_default = np.isclose(v, base)
        color = cmap((j + 0.5) / len(values))
        lw = 2.5 if is_default else 1.5
        alpha = 1.0 if is_default else 0.75
        suffix = " (default)" if is_default else ""
        ax.plot(t, traj, color=color, lw=lw, alpha=alpha, label=f"{pname}={v}{suffix}")
    ax.set_xlabel("time")
    ax.set_ylabel("fish biomass")
    ax.set_title(f"Sweep over {pname}")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Takeaways

- Without fishing, the Lotka-Volterra model **oscillates** around a neutrally stable centre. All orbits are closed; the centre attracts nothing.
- The phase portrait shows **concentric orbits** around $(A^*, F^*) = (\gamma/\delta, \alpha/\beta)$; the flow field reveals the counter-clockwise circulation.
- **Moderate harvest** shifts the equilibrium and keeps the system viable for a while. **Excessive harvest** drives the stock to zero and triggers `EcologyInstabilityError`.
- The harvest sweep quantifies the cliff: time-to-collapse drops abruptly past a critical $H$.
- The **L-V invariant** $V$ is conserved by exact dynamics. Euler **drifts**; RK45 keeps $V$ near-constant. The integrator choice is a substantive scientific decision, not a style preference.
- Each Greek parameter has an identifiable role: $\alpha$ and $\beta$ set the prey side, $\delta$ and $\gamma$ set the predator side. Equilibrium follows $F^* = \alpha/\beta$, $A^* = \gamma/\delta$.
- This module will be **wrapped as a Gymnasium environment** in **Brick 2**.